In [3]:
#1. Data loading
import pandas as pd

customers = pd.read_csv('../data/customers.csv')
invoice_items = pd.read_csv('../data/invoice_items.csv')
products = pd.read_csv('../data/products.csv')
purchases = pd.read_csv('../data/purchases.csv')

#dealing with paths has always been annoying i need to learn the best practices for this

In [4]:
print(type(customers))
#what type is this?
#what is a dataframe? 

<class 'pandas.DataFrame'>


In [5]:
#2. Data exploration

print("-"*20, "customers table", "-"*20)
print(customers.head()) #display the first 5 rows of the customers dataframe

#same for the other tables
print("-"*20, "invoice items table", "-"*20)
print(invoice_items.head())
print("-"*20, "products table", "-"*20)
print(products.head())
print("-"*20, "purchases table", "-"*20)
print(purchases.head())

-------------------- customers table --------------------
   CustomerID customer_type
0       17850    wholesaler
1       13047    wholesaler
2       12583    wholesaler
3       13748       private
4       15100       private
-------------------- invoice items table --------------------
   InvoiceID  product_id  quantity  price  line_total
0     536365        3851         6   2.55       15.30
1     536365        3859         6   3.39       20.34
2     536365         885         8   2.75       22.00
3     536365        1873         6   3.39       20.34
4     536365        2871         6   3.39       20.34
-------------------- products table --------------------
   product_id                                 item  \
0        3851   white hanging heart t-light holder   
1        3859                  white metal lantern   
2         885       cream cupid hearts coat hanger   
3        1873  knitted union flag hot water bottle   
4        2871       red woolly hottie white heart.   

      

In [6]:
print("rows then columns for customers:", customers.shape)

rows then columns for customers: (8237, 2)


In [7]:
#rows numbers in each
print("nbr of lines in customers:", customers.shape[0])
print("nbr of lines in invoice_items:", invoice_items.shape[0])
print("nbr of lines in products:", products.shape[0])
print("nbr of lines in purchases:", purchases.shape[0])

nbr of lines in customers: 8237
nbr of lines in invoice_items: 436689
nbr of lines in products: 4033
nbr of lines in purchases: 436689


In [8]:
print("-"*20, "customers table info", "-"*20)
customers.info() 
#get a summary of the dataframe
#including the number of non-null values 
# and the data types of each column
print("\n\n")
print("-"*20, "invoice items table info", "-"*20)
invoice_items.info()
print("\n\n")
print("-"*20, "products table info", "-"*20)
products.info()
print("\n\n")
print("-"*20, "purchases table info", "-"*20)
purchases.info()

-------------------- customers table info --------------------
<class 'pandas.DataFrame'>
RangeIndex: 8237 entries, 0 to 8236
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   CustomerID     8237 non-null   int64
 1   customer_type  8237 non-null   str  
dtypes: int64(1), str(1)
memory usage: 128.8 KB



-------------------- invoice items table info --------------------
<class 'pandas.DataFrame'>
RangeIndex: 436689 entries, 0 to 436688
Data columns (total 5 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   InvoiceID   436689 non-null  int64  
 1   product_id  436689 non-null  int64  
 2   quantity    436689 non-null  int64  
 3   price       436689 non-null  float64
 4   line_total  436689 non-null  float64
dtypes: float64(2), int64(3)
memory usage: 16.7 MB



-------------------- products table info --------------------
<class 'pandas.DataFrame'>
RangeIndex: 4033 ent

In [9]:
# Number of missing values per column
customers.isna().sum() #is NA?
#isna() returns a dataframe 
# of the same shape as customers, 
# with True for missing values 
# and False for non-missing values.

#how does it identify missing values?
#isna() identifies missing values as NaN (Not a Number) 
# or None values.

#what if the column has a space but nth else visually it is empty no?
#isna() will still identify it as a missing value if it's an empty string or NaN.

#why add sum() at the end?
#sum() is used to count the number of 
# True values in each column,
# which gives the total number of missing values per column.

CustomerID       0
customer_type    0
dtype: int64

In [10]:
customers.isna().sum()
invoice_items.isna().sum()
products.isna().sum()
purchases.isna().sum()


InvoiceID     0
date          0
CustomerID    0
product_id    0
quantity      0
dtype: int64

In [11]:
#Check for suspicious missing-value representations
"""
"" or whitespace
"N/A" or "n/a"
"unknown"
"null" or "None"
-1 or 999 used as a sentinel value
"""

#check text columns first
#customer type, item, category, date

#inspect the distinct values
customers['customer_type'].value_counts(dropna=False)
#value_counts() 
# counts the occurrences of each unique value 
# in the 'customer_type' column, 
# including NaN values if dropna=False.

#why inspect distinct values?
#Inspecting distinct values helps identify any unusual 
# or unexpected entries,
# which can indicate data quality issues or inconsistencies.

#look for blank strings and whitespace-only
customers['customer_type'].str.strip().eq("").sum()
#str.strip() removes leading and trailing whitespace 
# from each string in the 'customer_type' column.

#leading and trailing whitespace? 
#leading whitespace is any space, tab, or other whitespace character 
# that appears at the beginning of a string, 
# while trailing whitespace is any space, tab, or other whitespace character 
# that appears at the end of a string.

#eq("") checks if the stripped string is equal 
# to an empty string,
# and sum() counts the number of occurrences of empty strings.

#----so in summary 
#we remove spaces from the beginning and end of each string,
#then check if the resulting string is empty,
#then count how many empty strings there are


#check for common text placeholders
placeholders = ["N/A", "n/a", "unknown", "null", "None"]
customers['customer_type'].str.strip().isin(placeholders).sum()

print("Number of missing values in 'customer_type':", customers['customer_type'].isna().sum())
print("Number of blank strings in 'customer_type':", customers['customer_type'].str.strip().eq("").sum())
print("Number of placeholder values in 'customer_type':", customers['customer_type'].str.strip().isin(placeholders).sum())



Number of missing values in 'customer_type': 0
Number of blank strings in 'customer_type': 0
Number of placeholder values in 'customer_type': 0


In [12]:
# now check table by table all columns
#added astype("string") to avoid errors 
# when calling str.strip() on non-string columns

for col in customers.columns:
    print(f"Column: {col}")
    print("Number of missing values:", customers[col].isna().sum())
    print("Number of blank strings:", customers[col].astype("string").str.strip().eq("").sum())
    print("Number of placeholder values:", customers[col].astype("string").str.strip().isin(placeholders).sum())
    print("\n")

for col in invoice_items.columns:
    print(f"Column: {col}")
    print("Number of missing values:", invoice_items[col].isna().sum())
    print("Number of blank strings:", invoice_items[col].astype("string").str.strip().eq("").sum())
    print("Number of placeholder values:", invoice_items[col].astype("string").str.strip().isin(placeholders).sum())
    print("\n")

for col in products.columns:
    print(f"Column: {col}")
    print("Number of missing values:", products[col].isna().sum())
    print("Number of blank strings:", products[col].astype("string").str.strip().eq("").sum())
    print("Number of placeholder values:", products[col].astype("string").str.strip().isin(placeholders).sum())
    print("\n")

for col in purchases.columns:
    print(f"Column: {col}")
    print("Number of missing values:", purchases[col].isna().sum())
    print("Number of blank strings:", purchases[col].astype("string").str.strip().eq("").sum())
    print("Number of placeholder values:", purchases[col].astype("string").str.strip().isin(placeholders).sum())
    print("\n")

Column: CustomerID
Number of missing values: 0
Number of blank strings: 0
Number of placeholder values: 0


Column: customer_type
Number of missing values: 0
Number of blank strings: 0
Number of placeholder values: 0


Column: InvoiceID
Number of missing values: 0
Number of blank strings: 0
Number of placeholder values: 0


Column: product_id
Number of missing values: 0
Number of blank strings: 0
Number of placeholder values: 0


Column: quantity
Number of missing values: 0
Number of blank strings: 0
Number of placeholder values: 0


Column: price
Number of missing values: 0
Number of blank strings: 0
Number of placeholder values: 0


Column: line_total
Number of missing values: 0
Number of blank strings: 0
Number of placeholder values: 0


Column: product_id
Number of missing values: 0
Number of blank strings: 0
Number of placeholder values: 0


Column: item
Number of missing values: 0
Number of blank strings: 0
Number of placeholder values: 0


Column: category
Number of missing valu

In [13]:
#now checking for duplicates
# duplicate full rows or duplicate values in specific columns

print("Number of duplicate rows in customers:", customers.duplicated().sum())
# duplicated() returns a boolean Series 
# indicating which rows are duplicates

print("Number of duplicate rows in invoice_items:", invoice_items.duplicated().sum())
print("Number of duplicate rows in products:", products.duplicated().sum())
print("Number of duplicate rows in purchases:", purchases.duplicated().sum())

Number of duplicate rows in customers: 0
Number of duplicate rows in invoice_items: 5426
Number of duplicate rows in products: 0
Number of duplicate rows in purchases: 5516


In [14]:
invoice_items[invoice_items .duplicated(keep=False)]  # Display all duplicate rows in customers

,InvoiceID,product_id,quantity,price,line_total
476,536409,3040,1,4.95,4.95
480,536409,1608,1,2.10,2.10
485,536409,3669,1,1.25,1.25
508,536409,3669,1,1.25,1.25
512,536409,3062,1,2.95,2.95
...,...,...,...,...,...
436316,9976,1036,3,1.17,3.51
436478,3964,3637,1,3.73,3.73
436515,3964,3637,1,3.73,3.73
436638,13330,613,4,7.97,31.88


In [15]:
purchases[purchases.duplicated(keep=False)]  # Display all duplicate rows in purchases

,InvoiceID,date,CustomerID,product_id,quantity
476,536409,2014-12-01,17908,3040,1
480,536409,2014-12-01,17908,1608,1
485,536409,2014-12-01,17908,3669,1
508,536409,2014-12-01,17908,3669,1
512,536409,2014-12-01,17908,3062,1
...,...,...,...,...,...
436316,9976,2015-12-23,3679,1036,3
436478,3964,2015-12-27,2061,3637,1
436515,3964,2015-12-27,2061,3637,1
436638,13330,2015-12-30,4565,613,4


In [16]:
#check duplicates in customers no same id 
customers[customers.duplicated(subset=['CustomerID'], keep=False)]


,CustomerID,customer_type


In [17]:
#check duplicates in products no same id or name 
#meaning get all those that share same id or same name
products[products.duplicated(subset=['product_id', 'item'], keep=False)]

,product_id,item,category,price


In [18]:
#next for table of invoice_items check duplicates 
#get those that share same invoice_id and product_id
invoice_items[invoice_items.duplicated(subset=['InvoiceID', 'product_id'], keep=False)]

,InvoiceID,product_id,quantity,price,line_total
113,536381,2486,1,1.25,1.25
125,536381,2486,3,1.25,3.75
474,536409,84,3,6.35,19.05
476,536409,3040,1,4.95,4.95
480,536409,1608,1,2.10,2.10
...,...,...,...,...,...
436520,11306,2304,2,4.98,9.96
436588,4878,3891,4,2.19,8.76
436620,4878,3891,2,2.19,4.38
436638,13330,613,4,7.97,31.88


In [19]:
#next check purchases for same invoice and same product at same time
purchases[purchases.duplicated(subset=['InvoiceID', 'product_id'], keep=False)]


,InvoiceID,date,CustomerID,product_id,quantity
113,536381,2014-12-01,15311,2486,1
125,536381,2014-12-01,15311,2486,3
474,536409,2014-12-01,17908,84,3
476,536409,2014-12-01,17908,3040,1
480,536409,2014-12-01,17908,1608,1
...,...,...,...,...,...
436520,11306,2015-12-27,4039,2304,2
436588,4878,2015-12-29,2309,3891,4
436620,4878,2015-12-29,2309,3891,2
436638,13330,2015-12-30,4565,613,4


In [20]:
#next check purchases for same invoice and same product at same time but different dates only
# should show only rows with those specific conditions

purchases[purchases.groupby(['InvoiceID', 'product_id'])['date'].transform('nunique') > 1]

#groupby(['InvoiceID', 'product_id']) 
# groups the purchases  by InvoiceID and product_id

#['date'].transform('nunique')
# calculates the number of unique dates for each group,
# and assigns that value to each row in the group.


,InvoiceID,date,CustomerID,product_id,quantity


In [21]:
purchases.head(-20)

,InvoiceID,date,CustomerID,product_id,quantity
0,536365,2014-12-01,17850,3851,6
1,536365,2014-12-01,17850,3859,6
2,536365,2014-12-01,17850,885,8
3,536365,2014-12-01,17850,1873,6
4,536365,2014-12-01,17850,2871,6
...,...,...,...,...,...
436664,9540,2015-12-30,3562,2710,4
436665,6770,2015-12-30,2810,1341,3
436666,9597,2015-12-30,3579,1696,4
436667,2826,2015-12-30,1769,2281,3


In [22]:
#catch unusual values and outliers

customers.describe(include='all')
#describe() provides summary statistics 
# for numerical columns in the customers dataframe,
# including count, mean, standard deviation, minimum, 
# 25th percentile, 50th percentile (median), 75th percentile, 
# and maximum values.

,CustomerID,customer_type
count,8237.000000,8237
unique,NaN,2
top,NaN,private
freq,NaN,7161
mean,9480.631783,NaN
std,6316.159696,NaN
min,1000.000000,NaN
25%,3118.000000,NaN
50%,12618.000000,NaN
75%,15449.000000,NaN


In [23]:
products.describe()

,product_id,price
count,4033.000000,4033.000000
mean,2017.000000,3.798143
std,1164.371146,13.059295
min,1.000000,0.001000
25%,1009.000000,1.250000
50%,2017.000000,2.100000
75%,3025.000000,4.150000
max,4033.000000,649.500000


In [24]:
#find products with price less than 0.10 and greater than 0
products[(products["price"] > 0) & (products["price"] < 0.1)]

,product_id,item,category,price
1834,2705,porcelain budah incense holder,"Health, Beauty & Personal Care",0.070
2948,2370,pads to match all cushions,Home Decor,0.001


In [25]:
products.describe(include="object")

C:\Users\HIBA\AppData\Local\Temp\ipykernel_3296\2778178033.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  products.describe(include="object")


,item,category
count,4033,4033
unique,4033,25
top,white hanging heart t-light holder,Kitchen & Dining
freq,1,863


In [26]:
products["category"].value_counts(dropna=False)

category
Kitchen & Dining                  863
Home Decor                        758
Apparel & Accessories             642
Stationery & Craft                339
Toys & Games                      330
Health, Beauty & Personal Care    291
Party & Festive                   247
Seasonal                          215
Garden & Outdoor                  160
Miscellaneous                      39
Dairy                              18
Beverages                          17
Administrative                     16
Bakery & Snacks                    16
Produce                            15
Condiments & Spices                15
Meat                               11
Cleaning & Hygiene                 10
Personal Care                       7
Household                           6
DIY & Tools                         5
Pantry                              4
Gardening                           3
Pet Care                            3
Kitchen                             3
Name: count, dtype: int64

In [27]:
invoice_items.describe()

,InvoiceID,product_id,quantity,price,line_total
count,436689.000000,436689.000000,436689.000000,436689.00000,436689.000000
mean,511514.110834,2109.265131,12.087628,3.22677,21.374055
std,157823.706391,1137.512313,172.252472,21.11145,295.049213
min,1.000000,1.000000,1.000000,0.00000,0.000000
25%,545589.000000,1171.000000,2.000000,1.25000,4.680000
50%,559545.000000,2093.000000,4.000000,2.08000,10.760000
75%,571229.000000,3062.000000,12.000000,3.75000,18.990000
max,581587.000000,4033.000000,80995.000000,8142.75000,168469.600000


In [28]:
invoice_items[invoice_items["price"] < 0.10]

,InvoiceID,product_id,quantity,price,line_total
6914,537197,2983,1,0.00,0.00
10370,537618,2705,100,0.07,7.00
22539,539263,125,4,0.00,0.00
25379,539722,2876,10,0.00,0.00
29080,540372,2404,24,0.00,0.00
...,...,...,...,...,...
387655,580635,202,11,0.09,0.99
387747,580639,645,10,0.06,0.60
390609,580822,645,80,0.06,4.80
390610,580822,1972,80,0.06,4.80


In [29]:
invoice_items[invoice_items["quantity"] > 10000]

,InvoiceID,product_id,quantity,price,line_total
37126,541431,2074,74215,1.04,77183.6
370473,578841,239,12540,0.00,0.0
397451,581483,2414,80995,2.08,168469.6


In [30]:
products[products["product_id"] == 2414]

,product_id,item,category,price
3865,2414,"paper craft , little birdie",Stationery & Craft,2.08


In [31]:
#how many products have different price in tables products and invoice_items? and show both prices price_product and price_invoice for comparison

#merge two tables
price_comparison = pd.merge(
    products[["product_id", "item", "price"]],
    invoice_items[["product_id", "price"]],
    on="product_id",
    suffixes=("_product", "_invoice")
)

#find
different_prices = price_comparison[
    price_comparison["price_product"] !=
    price_comparison["price_invoice"]
]

len(different_prices) # number of invoice-item records with different prices

unique_different_prices = (
    different_prices[["product_id", "item", "price_product", "price_invoice"]]
    .drop_duplicates()
    .sort_values(["product_id", "price_product", "price_invoice"])
)
display(unique_different_prices)

,product_id,item,price_product,price_invoice
96883,1,10 colour spaceboy pen,0.85,0.72
96954,1,10 colour spaceboy pen,0.85,1.63
224743,2,12 coloured party balloons,0.65,1.25
300543,4,12 egg house painted wood,12.75,10.95
300539,4,12 egg house painted wood,12.75,24.96
...,...,...,...,...
40534,4030,zinc willie winkie candle stick,0.85,0.72
40539,4030,zinc willie winkie candle stick,0.85,1.63
333615,4031,zinc wire kitchen organiser,10.40,3.95
331598,4032,zinc wire sweetheart letter tray,3.75,1.25


In [32]:
#check if line_total in invoice_items is equal to quantity * price to make sure it was calculated correctly
line_totals = invoice_items['quantity'] * invoice_items['price']

mismatched_line_totals = invoice_items[
    invoice_items['line_total'].round(2) != line_totals.round(2)
].copy()

mismatched_line_totals['calculated_line_total'] = line_totals.loc[
    mismatched_line_totals.index
].round(2)

display(mismatched_line_totals)

,InvoiceID,product_id,quantity,price,line_total,calculated_line_total


In [33]:
#show top 10 maximum prices in invoice_items
top_max_prices = invoice_items.nlargest(10, 'price')
display(top_max_prices)

,InvoiceID,product_id,quantity,price,line_total
118352,551697,2711,1,8142.75,8142.75
308822,573077,2061,1,4161.06,4161.06
308846,573080,2061,1,4161.06,4161.06
295254,571751,2061,1,3949.32,3949.32
269591,569382,2061,1,3155.95,3155.95
208732,562946,2061,1,2500.00,2500.00
97061,548813,2061,1,2382.92,2382.92
295255,571751,2061,1,2118.74,2118.74
97064,548820,2061,1,2053.07,2053.07
244129,566927,2061,1,2033.10,2033.10


In [34]:
#calculate how many different prices are there for each product in invoice_items
price_counts = invoice_items.groupby('product_id')['price'].nunique().reset_index()
#display them as well as a column with a list of the different unique prices for each product
price_counts_with_list = (
    invoice_items.groupby('product_id')['price']
    .agg(nunique='nunique', unique_prices=lambda prices: prices.unique().tolist())
    .reset_index()
)


#add column with price from products table and a column that says yes or no if the price exists in the list of unique prices for that product in invoice_items
price_counts_with_list = price_counts_with_list.merge(
    products[["product_id", "price"]].rename(columns={"price": "product_price"}),
    on="product_id",
    how="left"
)
price_counts_with_list["product_price_in_invoice_prices"] = price_counts_with_list.apply(
    lambda row: row["product_price"] in row["unique_prices"],
    axis=1
)
display(price_counts_with_list)

# count how many no? 
no_count = price_counts_with_list["product_price_in_invoice_prices"].value_counts().get(False, 0)
print(f"Number of products where the product price is not in the list of unique prices in invoice_items: {no_count}")


,product_id,nunique,unique_prices,product_price,product_price_in_invoice_prices
0,1,3,"[0.85, 0.72, 1.63]",0.85,True
1,2,2,"[0.65, 1.25]",0.65,True
2,3,1,[1.65],1.65,True
3,4,3,"[12.75, 24.96, 10.95]",12.75,True
4,5,1,[2.08],2.08,True
...,...,...,...,...,...
4028,4029,1,[16.95],16.95,True
4029,4030,3,"[0.85, 0.72, 1.63]",0.85,True
4030,4031,2,"[10.4, 3.95]",10.40,True
4031,4032,3,"[3.75, 3.29, 1.25]",3.75,True


Number of products where the product price is not in the list of unique prices in invoice_items: 0


In [35]:
#next examining purchases table for unusual values and outliers
purchases.describe()

,InvoiceID,CustomerID,product_id,quantity
count,436689.000000,436689.000000,436689.000000,436689.000000
mean,511514.110834,14203.268795,2109.265131,12.087628
std,157823.706391,3874.517869,1137.512313,172.252472
min,1.000000,1000.000000,1.000000,1.000000
25%,545589.000000,13384.000000,1171.000000,2.000000
50%,559545.000000,14911.000000,2093.000000,4.000000
75%,571229.000000,16705.000000,3062.000000,12.000000
max,581587.000000,18287.000000,4033.000000,80995.000000


In [36]:
"""
# Check for duplicate combinations in both tables
print("Duplicates in purchases:",
      purchases.duplicated(["InvoiceID", "product_id"]).sum())

print("Duplicates in invoice_items:",
      invoice_items.duplicated(["InvoiceID", "product_id"]).sum())


# Merge the two tables using InvoiceID and product_id
merged_purchases_invoice = purchases.merge(
    invoice_items[["InvoiceID", "product_id", "quantity"]],
    on=["InvoiceID", "product_id"],
    how="left",
    validate="one_to_one"
)


# Compare quantities between the two tables
mismatched_count = (
    merged_purchases_invoice["quantity_x"] !=
    merged_purchases_invoice["quantity_y"]
)


# Count mismatched rows
print("Number of mismatched quantities:", mismatched_count.sum())


# Display only rows where quantities are different
mismatched_quantities = merged_purchases_invoice.loc[
    mismatched_count,
    ["InvoiceID", "product_id", "quantity_x", "quantity_y"]
].copy()

# Rename columns for clarity
mismatched_quantities.columns = [
    "InvoiceID",
    "product_id",
    "quantity_purchases",
    "quantity_invoice_items"
]

display(mismatched_quantities)
"""

#doesnt work for now maybe cause of duplicates and stuff i will try again later

'\n# Check for duplicate combinations in both tables\nprint("Duplicates in purchases:",\n      purchases.duplicated(["InvoiceID", "product_id"]).sum())\n\nprint("Duplicates in invoice_items:",\n      invoice_items.duplicated(["InvoiceID", "product_id"]).sum())\n\n\n# Merge the two tables using InvoiceID and product_id\nmerged_purchases_invoice = purchases.merge(\n    invoice_items[["InvoiceID", "product_id", "quantity"]],\n    on=["InvoiceID", "product_id"],\n    how="left",\n    validate="one_to_one"\n)\n\n\n# Compare quantities between the two tables\nmismatched_count = (\n    merged_purchases_invoice["quantity_x"] !=\n    merged_purchases_invoice["quantity_y"]\n)\n\n\n# Count mismatched rows\nprint("Number of mismatched quantities:", mismatched_count.sum())\n\n\n# Display only rows where quantities are different\nmismatched_quantities = merged_purchases_invoice.loc[\n    mismatched_count,\n    ["InvoiceID", "product_id", "quantity_x", "quantity_y"]\n].copy()\n\n# Rename columns for 

In [37]:
#check dates
# Convert dates using the expected format
import pandas as pd

dates = pd.to_datetime(purchases['date'], format='%Y-%m-%d', errors='coerce')

# Find invalid dates
invalid_dates = purchases[dates.isna()]

# Display results
print("Number of invalid dates:", len(invalid_dates))
display(invalid_dates)

Number of invalid dates: 0


,InvoiceID,date,CustomerID,product_id,quantity


In [42]:
# Check unusually large quantities (> 100) by customer type

# Get one customer per invoice
invoice_customers = purchases[["InvoiceID", "CustomerID"]].drop_duplicates()

# Join invoice items with customer information
all_items = (
    invoice_items
    .merge(
        invoice_customers,
        on="InvoiceID",
        how="left",
        validate="many_to_one"
    )
    .merge(
        customers[["CustomerID", "customer_type"]],
        on="CustomerID",
        how="left",
        validate="many_to_one"
    )
)

# Filter quantities above 100
large_quantities = all_items[all_items["quantity"] > 100].copy()

# Summary by customer type
summary = large_quantities.groupby("customer_type").agg(
    number_of_lines=("quantity", "count"),
    total_quantity=("quantity", "sum"),
    average_quantity=("quantity", "mean"),
    max_quantity=("quantity", "max")
)

# Percentage of invoice lines above 100 for each customer type
percentages = (
    all_items.assign(is_large=all_items["quantity"] > 100)
    .groupby("customer_type")["is_large"]
    .mean()
    .mul(100)
    .round(2)
)

summary["percentage_of_customer_lines"] = percentages

print(f"Total invoice lines: {len(all_items)}")
print(f"Invoice lines above 100: {len(large_quantities)}")

display(summary)

Total invoice lines: 436689
Invoice lines above 100: 4665


,number_of_lines,total_quantity,average_quantity,max_quantity,percentage_of_customer_lines
customer_type,,,,,
private,275,48976,178.094545,600,0.15
wholesaler,4390,1346846,306.798633,80995,1.74


In [43]:
admin = products[products["category"] == "Administrative"]
print(admin[["product_id", "item", "price"]])

# the rows in invoice_items that use them
invoice_items[invoice_items["product_id"].isin(admin["product_id"])] \
    .sort_values("price", ascending=False).head(10)

      product_id                             item  price
45          2711                          postage  18.00
773          643                         carriage  50.00
855         2061                           manual   1.25
1216         272                     bank charges  15.00
1346         548         brown check cat doorstop   4.25
1636        2385   pair padded hangers pink check   3.75
1696        3514             sunset check hammock   7.95
2489         414      blue crusoe check lampshade   0.42
2731        1075         dotcomgiftshop tea towel   1.65
3009         408  blue check bag w handle 34x20cm   0.83
3144        1356          french carriage lantern   6.65
3173        2238                next day carriage  15.00
3316         339     black baroque carriage clock   9.95
3368        2369                   packing charge   7.50
3514        1074                   dotcom postage  11.17
3751        1698            high resolution image   3.00


,InvoiceID,product_id,quantity,price,line_total
118352,551697,2711,1,8142.75,8142.75
308846,573080,2061,1,4161.06,4161.06
308822,573077,2061,1,4161.06,4161.06
295254,571751,2061,1,3949.32,3949.32
269591,569382,2061,1,3155.95,3155.95
208732,562946,2061,1,2500.00,2500.00
97061,548813,2061,1,2382.92,2382.92
295255,571751,2061,1,2118.74,2118.74
97064,548820,2061,1,2053.07,2053.07
244129,566927,2061,1,2033.10,2033.10


In [49]:
# check keywords for categories

from collections import Counter
import re

# Common words that don't help identify products
stopwords = {
    # Existing filler words
    "the", "and", "for", "with", "of", "in", "to",
    "a", "an", "on", "set", "pack",

    # Colors (including UK spellings)
    "black", "white", "red", "blue", "green", "yellow",
    "pink", "purple", "orange", "brown", "grey", "gray",
    "gold", "silver", "cream", "beige", "clear", "colour", "ivory",
    "color", "coloured", "colored", 

    # Common descriptive terms
    "small", "medium", "large", "mini", "big",
    "new", "vintage", "assorted", "various", "heart"
    "metal", "plastic", "wooden"
}

for category in products["category"].dropna().unique():

    category_products = products[
        products["category"] == category
    ]["item"].dropna()

    words = []

    for item in category_products:
        item_words = re.findall(r"\b[a-zA-Z]+\b", item.lower())
        words.extend(
            word for word in item_words
            if word not in stopwords and len(word) > 2
        )

    print(f"\n--- {category} ---")
    print(Counter(words).most_common(20))


--- Home Decor ---
[('heart', 101), ('wall', 73), ('holder', 65), ('light', 62), ('metal', 58), ('art', 55), ('box', 42), ('hanging', 40), ('sign', 40), ('frame', 39), ('mirror', 39), ('candle', 37), ('clock', 30), ('photo', 30), ('rose', 27), ('flower', 26), ('french', 22), ('bird', 21), ('candles', 21), ('cover', 21)]

--- Health, Beauty & Personal Care ---
[('bottle', 29), ('hot', 27), ('incense', 25), ('water', 24), ('tin', 19), ('candle', 19), ('doiley', 19), ('rose', 18), ('heart', 17), ('design', 17), ('bag', 16), ('doily', 16), ('scented', 14), ('tissues', 14), ('towel', 14), ('bathroom', 13), ('hand', 12), ('tea', 11), ('paper', 11), ('soap', 11)]

--- Party & Festive ---
[('decoration', 70), ('party', 41), ('garland', 32), ('heart', 30), ('hanging', 19), ('ribbon', 19), ('reel', 18), ('bunting', 18), ('stickers', 16), ('ribbons', 14), ('paper', 14), ('flag', 11), ('embroidered', 10), ('union', 9), ('hearts', 9), ('zinc', 8), ('polkadot', 8), ('metal', 8), ('bells', 8), ('hon